# Internal Heat Gains (IHG) Equations

This notebook summarizes and computes the heat-gain equations from the provided source text.

## 1) Total Heat Gains

The total heat gains include gains from sun, lighting, people, machines, devices, infiltration, and adjacent rooms:

$$Q = Q_W + Q_{WS} + Q_L + Q_P + Q_E + Q_D + Q_{In} + Q_B$$

Where:
- $Q_W$: gains from sun via transparent barriers (windows) [W]
- $Q_{WS}$: gains from sun via non-transparent barriers (walls) [W]
- $Q_L$: heat gains from lighting [W]
- $Q_P$: heat gains from people [W]
- $Q_E$: heat gains from electrical engines/machines [W]
- $Q_D$: heat gains from other devices [W]
- $Q_{In}$: heat gains due to air infiltration [W]
- $Q_B$: gains through barriers of adjacent rooms [W]

In [1]:
def total_heat_gain(QW, QWS, QL, QP, QE, QD, QIn, QB):
    return QW + QWS + QL + QP + QE + QD + QIn + QB

## 2) Heat Gains From People

People gains are comprised of sensible and latent components.

### 2.1 Sensible Heat Gains From People

$$Q_{P,s} = \varphi \cdot n \cdot q_j$$

Where:
- $\varphi$: people residence coincidence factor (0.4 to 1.0)
- $n$: number of people
- $q_j$: unit sensible heat flow delivered by one person [W/person]

In [2]:
def sensible_people_gain(phi, n, qj):
    """Sensible heat gain from people [W]."""
    return phi * n * qj

### 2.2 Latent (Humidity) Gains From People

Water vapor generation is:

$$W_{P} = \varphi \cdot n \cdot w_j$$

Where:
- $w_j$: unit water vapor flow from one person [g/h per person]
- $W_P$: total water vapor gain [g/h]

Optional conversion to latent heat power (if needed):

$$Q_{P,l} = \dot m_w \cdot h_{fg}$$

with $\dot m_w$ in kg/s and $h_{fg}$ in J/kg.

In [3]:
def latent_people_moisture_gain(phi, n, wj):
    """Moisture gain from people [g/h]."""
    return phi * n * wj

def moisture_gh_to_latent_watts(Wp_gh, h_fg=2.45e6):
    """Convert moisture flow [g/h] to latent heat [W].
    h_fg default is ~2.45e6 J/kg near room temperature.
    """
    m_dot_kg_s = (Wp_gh / 1000.0) / 3600.0
    return m_dot_kg_s * h_fg

## 3) Table-Based Inputs

From the provided method, these are generally taken from simplified HVAC tables (based on activity, room temperature, lighting type, and device type):
- $q_j$ (sensible W/person)
- $w_j$ (moisture g/h-person)
- lighting heat gains
- device and electrical equipment gains
- solar gains through transparent/non-transparent barriers

Notes from source:
- For women: decrease tabulated $q_j$ and $w_j$ values by 20%.
- For children: decrease by 20-40% depending on age.
- In small buildings and in theatres/cinemas, $\varphi = 1$.

## 4) Example Calculation

Edit the values below with your project-specific data.

In [4]:
# Example inputs (replace with your real values)
QW = 500.0
QWS = 300.0
QL = 450.0
QE = 600.0
QD = 250.0
QIn = 200.0
QB = 150.0

phi = 0.8
n = 30
qj = 75.0      # W/person
wj = 55.0      # g/h-person

QP_s = sensible_people_gain(phi, n, qj)
Wp = latent_people_moisture_gain(phi, n, wj)
QP_l = moisture_gh_to_latent_watts(Wp)
QP = QP_s + QP_l

Q_total = total_heat_gain(QW, QWS, QL, QP, QE, QD, QIn, QB)

print(f'Sensible people gain QP_s: {QP_s:.2f} W')
print(f'Latent people moisture Wp: {Wp:.2f} g/h')
print(f'Latent people gain QP_l: {QP_l:.2f} W')
print(f'Total people gain QP: {QP:.2f} W')
print(f'Total heat gain Q: {Q_total:.2f} W')

Sensible people gain QP_s: 1800.00 W
Latent people moisture Wp: 1320.00 g/h
Latent people gain QP_l: 898.33 W
Total people gain QP: 2698.33 W
Total heat gain Q: 5148.33 W


## 5) Carbon Dioxide Generation Rates for Occupants (Persily & de Jonge, 2017)

This section adds the key CO2 equations used in IAQ/ventilation calculations.

### 5.1 Ventilation and Indoor CO2 Steady-State

$$Q_o = \frac{G}{C_{in,ss} - C_{out}} \quad (1)$$

Where:
- $Q_o$: outdoor air ventilation rate per person
- $G$: CO2 generation rate per person
- $C_{in,ss}$: steady-state indoor CO2 concentration
- $C_{out}$: outdoor CO2 concentration

### 5.2 Legacy ASHRAE/ASTM Approach

$$V_{O_2} = \frac{0.00276\,A_D\,M}{0.23\,RQ + 0.77} \quad (2)$$

$$A_D = 0.202\,H^{0.725}\,W^{0.425} \quad (3)$$

$$V_{CO_2} = V_{O_2}\,RQ = \frac{0.00276\,A_D\,M\,RQ}{0.23\,RQ + 0.77} \quad (4)$$

Where:
- $A_D$: DuBois body surface area [m$^2$]
- $H$: height [m]
- $W$: body mass [kg]
- $M$: activity level [met]
- $RQ$: respiratory quotient (often ~0.85)

### 5.3 Diet-Based RQ (Food Quotient)

$$FQ = 1.0\,CA + 0.7\,F + 0.79\,P + 0.66\,A \quad (7)$$

Where $CA$, $F$, $P$, and $A$ are fractions of dietary energy from carbohydrates, fat, protein, and alcohol (sum to 1.0).

### 5.4 BMR-Based Updated Approach

At 273 K and 101 kPa (with BMR in MJ/day and $M$ in met):

$$V_{CO_2} = RQ\,BMR\,M\,(0.000569) \quad (8)$$

Assuming $RQ=0.85$:

$$V_{CO_2} = BMR\,M\,(0.000484) \quad (9)$$

For general pressure/temperature conditions:

$$V_{CO_2} = RQ\,BMR\,M\,\left(\frac{T}{P}\right)(0.000211) \quad (10)$$

Assuming $RQ=0.85$:

$$V_{CO_2} = BMR\,M\,\left(\frac{T}{P}\right)(0.000179) \quad (11)$$

Where $T$ is temperature [K] and $P$ is pressure [kPa].

In [5]:
# CO2 generation helper functions
def co2_steady_state_ventilation_per_person(G, Cin_ss, Cout):
    """Equation (1): Qo = G / (Cin_ss - Cout)."""
    return G / (Cin_ss - Cout)

def dubois_area(H_m, W_kg):
    """Equation (3): DuBois body surface area [m^2]."""
    return 0.202 * (H_m ** 0.725) * (W_kg ** 0.425)

def vo2_legacy(AD, M, RQ=0.85):
    """Equation (2): oxygen consumption [L/s-person]."""
    return (0.00276 * AD * M) / (0.23 * RQ + 0.77)

def vco2_legacy(H_m, W_kg, M, RQ=0.85):
    """Equation (4): CO2 generation from legacy method [L/s-person]."""
    AD = dubois_area(H_m, W_kg)
    return vo2_legacy(AD, M, RQ) * RQ

def food_quotient(CA, F, P, A):
    """Equation (7): diet-based FQ (often used as an RQ proxy)."""
    return 1.0 * CA + 0.7 * F + 0.79 * P + 0.66 * A

def vco2_bmr_273k_101kpa(BMR_MJ_day, M, RQ=0.85):
    """Equation (8): CO2 generation [L/s-person] at 273 K, 101 kPa."""
    return RQ * BMR_MJ_day * M * 0.000569

def vco2_bmr_273k_101kpa_rq085(BMR_MJ_day, M):
    """Equation (9): simplified form for RQ=0.85."""
    return BMR_MJ_day * M * 0.000484

def vco2_bmr_general(BMR_MJ_day, M, T_K, P_kPa, RQ=0.85):
    """Equation (10): general P/T conditions [L/s-person]."""
    return RQ * BMR_MJ_day * M * (T_K / P_kPa) * 0.000211

def vco2_bmr_general_rq085(BMR_MJ_day, M, T_K, P_kPa):
    """Equation (11): general P/T with RQ=0.85 [L/s-person]."""
    return BMR_MJ_day * M * (T_K / P_kPa) * 0.000179


In [6]:
# Example from paper narrative (adult office-like activity)
BMR = 7.73      # MJ/day
M = 1.5         # met
RQ = 0.85

vco2_std = vco2_bmr_273k_101kpa(BMR, M, RQ)
vco2_general_std = vco2_bmr_general(BMR, M, T_K=273.0, P_kPa=101.0, RQ=RQ)

print(f'VCO2 at 273 K & 101 kPa (Eq. 8): {vco2_std:.6f} L/s-person')
print(f'VCO2 from Eq. 10 at same conditions: {vco2_general_std:.6f} L/s-person')
print(f'Legacy Eq. 4 example (H=1.75 m, W=85 kg, M=1.5): {vco2_legacy(1.75, 85.0, M, RQ):.6f} L/s-person')

VCO2 at 273 K & 101 kPa (Eq. 8): 0.005608 L/s-person
VCO2 from Eq. 10 at same conditions: 0.005621 L/s-person
Legacy Eq. 4 example (H=1.75 m, W=85 kg, M=1.5): 0.007298 L/s-person


### 5.5 Schofield BMR Equations (MJ/day)

For body mass $m$ [kg]:

Males:
- $<3$: $BMR = 0.249m - 0.127$
- $3$ to $10$: $BMR = 0.095m + 2.110$
- $10$ to $18$: $BMR = 0.074m + 2.754$
- $18$ to $30$: $BMR = 0.063m + 2.896$
- $30$ to $60$: $BMR = 0.048m + 3.653$
- $\ge 60$: $BMR = 0.049m + 2.459$

Females:
- $<3$: $BMR = 0.244m - 0.130$
- $3$ to $10$: $BMR = 0.085m + 2.033$
- $10$ to $18$: $BMR = 0.056m + 2.898$
- $18$ to $30$: $BMR = 0.062m + 2.036$
- $30$ to $60$: $BMR = 0.034m + 3.538$
- $\ge 60$: $BMR = 0.038m + 2.755$

In [7]:
def schofield_bmr_mj_day(sex, age_years, mass_kg):
    """Estimate BMR [MJ/day] using Schofield equations from the paper."""
    s = sex.strip().lower()
    if s not in {'male', 'female'}:
        raise ValueError("sex must be 'male' or 'female'")

    if s == 'male':
        if age_years < 3: return 0.249 * mass_kg - 0.127
        if age_years < 10: return 0.095 * mass_kg + 2.110
        if age_years < 18: return 0.074 * mass_kg + 2.754
        if age_years < 30: return 0.063 * mass_kg + 2.896
        if age_years < 60: return 0.048 * mass_kg + 3.653
        return 0.049 * mass_kg + 2.459

    if age_years < 3: return 0.244 * mass_kg - 0.130
    if age_years < 10: return 0.085 * mass_kg + 2.033
    if age_years < 18: return 0.056 * mass_kg + 2.898
    if age_years < 30: return 0.062 * mass_kg + 2.036
    if age_years < 60: return 0.034 * mass_kg + 3.538
    return 0.038 * mass_kg + 2.755

# Quick check from paper narrative (approximate values)
print('Male, 85 kg, age 35 BMR [MJ/day]:', round(schofield_bmr_mj_day('male', 35, 85), 3))
print('Female, 75 kg, age 35 BMR [MJ/day]:', round(schofield_bmr_mj_day('female', 35, 75), 3))


Male, 85 kg, age 35 BMR [MJ/day]: 7.733
Female, 75 kg, age 35 BMR [MJ/day]: 6.088


### 5.6 Oxidation/RQ Reference Equations

Carbohydrate (glucose):
$$6O_2 + C_6H_{12}O_6 \rightarrow 6CO_2 + 6H_2O + 2760\,kJ, \quad RQ = 1.0$$

Fat (palmitic acid):
$$23O_2 + C_{16}H_{32}O_2 \rightarrow 16CO_2 + 16H_2O + 11090\,kJ, \quad RQ = 0.7$$

## 6) Moisture Generation Equations (People, Pets, Plants)

This section adds moisture-source equations and design values from the provided ASHRAE-style text.

### 6.1 People (Respiration + Transpiration)

Given indoor air at 20 to 24 C, the provided relation for expired air humidity ratio is:

$$(W_e - W_i) = A + B t_i - 0.798 W_i \quad (1)$$

Where:
- $W_e$: humidity ratio of expired air
- $W_i$: humidity ratio of indoor air
- $t_i$: indoor air temperature
- For $t_i$ in C: $A=0.02760$, $B=0.0000650$
- For $t_i$ in F: $A=0.02645$, $B=0.0000361$

Respiration airflow rate (adult reference):
- $\dot V_{resp} \approx 240$ L/h per m$^2$ body surface area
- Adult reference area: $A_{body} \approx 1.8$ m$^2$

Design notes from provided text:
- Total adult moisture release at rest is commonly treated as ~50 g/h (humidity-independent for design)
- Resting adult range: ~30 to 70 g/h (0.8 to 1.7 kg/day)
- Body-weight scaling approximation: moisture release proportional to body mass

### 6.2 Plant/Surface Evaporation Equations

Open-water style relation (no wind):

$$E = A (p_{w,s} - p_w) \quad (2)$$

Where:
- $E$: evaporation flux [kg/(s m$^2$)]
- $A$: mass transfer coefficient (SI: $3.5\times10^{-8}$)
- $p_{w,s}$: saturation vapor pressure [Pa]
- $p_w$: ambient vapor pressure [Pa]

Alternative linear concentration-gradient form:

$$J = B (C_w - C_a) \quad (3)$$

Where:
- $J$: evaporation flux [kg/(s m$^2$)]
- $B$: mass transfer coefficient
- $C_w$: water-vapor concentration at source surface [kg/m$^3$]
- $C_a$: ambient water-vapor concentration [kg/m$^3$]

### 6.3 Representative Design Values

- Adult at rest (design): 50 g/h
- Cat: about 2 g/h
- Other pets: about 0.7 g/h-kg body mass
- Houseplant: literature range includes ~0.84 to 2.5 g/h per plant (higher values also reported depending on plant type/conditions)

Residential moisture generation rates from provided table (TenWolde & Walker, 2001):
- 1 bedroom / 2 occupants: 8 L/day
- 2 bedroom / 3 occupants: 12 L/day
- 3 bedroom / 4 occupants: 14 L/day
- 4 bedroom / 5 occupants: 15 L/day
- Additional bedroom/occupant: +1 L/day

In [8]:
# Moisture generation helper functions
def humidity_ratio_delta_respiration(Wi, ti, units='C'):
    """Equation (1): returns (We - Wi)."""
    u = units.upper()
    if u == 'C':
        A, B = 0.02760, 0.0000650
    elif u == 'F':
        A, B = 0.02645, 0.0000361
    else:
        raise ValueError("units must be 'C' or 'F'")
    return A + B * ti - 0.798 * Wi

def respiration_moisture_gph(Wi, ti, body_surface_area_m2=1.8, units='C'):
    """Approximate respiration moisture generation [g/h].
    Uses Eq. (1) and respiration flow ~240 L/h per m^2 body surface area.
    Assumes humidity ratio in kg water / kg dry air and rho_dry_air ~1.2 kg/m^3.
    """
    dW = humidity_ratio_delta_respiration(Wi, ti, units=units)
    Vdot_m3_h = 0.240 * body_surface_area_m2  # 240 L/h/m^2 = 0.240 m^3/h/m^2
    mdot_dry_air_kg_h = 1.2 * Vdot_m3_h
    mdot_water_kg_h = dW * mdot_dry_air_kg_h
    return mdot_water_kg_h * 1000.0

def person_moisture_gph_design(num_people, gph_per_person=50.0):
    """Design moisture generation from people [g/h]."""
    return num_people * gph_per_person

def person_moisture_by_weight_gph(body_mass_kg, reference_adult_kg=70.0, adult_rate_gph=50.0):
    """Weight-proportional estimate [g/h] from provided scaling concept."""
    return adult_rate_gph * (body_mass_kg / reference_adult_kg)

def pet_moisture_gph(body_mass_kg, rate_gph_per_kg=0.7):
    """Pet moisture generation [g/h]."""
    return rate_gph_per_kg * body_mass_kg

def evaporation_flux_pressure(pa_sat, pa_air, A=3.5e-8):
    """Equation (2): evaporation flux E [kg/(s*m^2)]."""
    return A * (pa_sat - pa_air)

def evaporation_flux_concentration(Cw, Ca, B):
    """Equation (3): evaporation flux J [kg/(s*m^2)]."""
    return B * (Cw - Ca)

def moisture_from_flux_gph(flux_kg_s_m2, area_m2):
    """Convert flux [kg/(s*m^2)] and area [m^2] to [g/h]."""
    return flux_kg_s_m2 * area_m2 * 3600.0 * 1000.0

def residential_moisture_L_day(num_bedrooms):
    """Piecewise design table from provided text (TenWolde & Walker, 2001)."""
    if num_bedrooms < 1:
        raise ValueError('num_bedrooms must be >= 1')
    base = {1: 8, 2: 12, 3: 14, 4: 15}
    if num_bedrooms in base:
        return base[num_bedrooms]
    return 15 + (num_bedrooms - 4) * 1


In [9]:
# Example calculations
Wi = 0.008   # indoor humidity ratio [kg/kg dry air] (example)
ti = 21.1    # C

dW = humidity_ratio_delta_respiration(Wi, ti, units='C')
resp_gph = respiration_moisture_gph(Wi, ti, body_surface_area_m2=1.8, units='C')
people_gph = person_moisture_gph_design(num_people=4, gph_per_person=50.0)
cat_gph = pet_moisture_gph(body_mass_kg=2.5)
dog_gph = pet_moisture_gph(body_mass_kg=10.0)
res_l_day = residential_moisture_L_day(3)

print(f'(We - Wi) from Eq. (1): {dW:.6f} kg/kg dry air')
print(f'Respiration moisture (adult, approximate): {resp_gph:.2f} g/h')
print(f'People design moisture (4 adults @50 g/h): {people_gph:.1f} g/h')
print(f'Cat estimate (2.5 kg): {cat_gph:.2f} g/h')
print(f'Dog estimate (10 kg): {dog_gph:.2f} g/h')
print(f'Residential design moisture (3 bedrooms): {res_l_day} L/day')

(We - Wi) from Eq. (1): 0.022587 kg/kg dry air
Respiration moisture (adult, approximate): 11.71 g/h
People design moisture (4 adults @50 g/h): 200.0 g/h
Cat estimate (2.5 kg): 1.75 g/h
Dog estimate (10 kg): 7.00 g/h
Residential design moisture (3 bedrooms): 14 L/day


## 7) Occupancy Estimation at Each Measurement (Room 354)

This section uses the Room 354 data and the CO2 mass-balance steady-state form to estimate people count at each timestamp.

Model used:
$$n(t) = \frac{Q(t)\,[C_{in}(t)-C_{out}]}{G_{person}}$$

with:
- $Q(t)$: outdoor/supply airflow [L/s]
- $C_{in}(t)$ and $C_{out}$ in ppm(v), using $\Delta C/10^6$ as volume fraction
- $G_{person}$: per-person CO2 generation [L/s-person] from Eq. (8)/(9)

Assumptions are exposed below and can be edited.

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

fpb_path = Path('/workspaces/CSC_4260_project/data/354_FPB_30day(03-28-2026).csv')
iaq_path = Path('/workspaces/CSC_4260_project/data/354_IAQ_30day(03-28-2026).csv')

# 1) Load FPB airflow
fpb = pd.read_csv(fpb_path)
fpb['ts'] = pd.to_datetime(fpb['dateTimeUtc'])
flow = (fpb[fpb['pointDisplayName'].eq('Discharge Air Flow')][['ts','value']]
        .rename(columns={'value':'flow_cfm'})
        .sort_values('ts'))

# 2) Load IAQ CO2 (primary) and FPB Zone CO2 (fallback)
iaq = pd.read_csv(iaq_path)
iaq['ts'] = pd.to_datetime(iaq['dateTimeUtc'])
co2_iaq = (iaq[iaq['pointDisplayName'].eq('CO2')][['ts','value']]
           .rename(columns={'value':'co2_iaq_ppm'})
           .sort_values('ts'))

co2_fpb = (fpb[fpb['pointDisplayName'].eq('Zone CO2')][['ts','value']]
           .rename(columns={'value':'co2_fpb_ppm'})
           .sort_values('ts'))

# 3) Time align (nearest within 2 minutes)
base = flow.copy()
base = pd.merge_asof(base, co2_iaq, on='ts', direction='nearest', tolerance=pd.Timedelta('2min'))
base = pd.merge_asof(base, co2_fpb, on='ts', direction='nearest', tolerance=pd.Timedelta('2min'))

# 4) Choose CO2 signal: IAQ first, FPB as fallback
base['co2_ppm_raw'] = base['co2_iaq_ppm'].combine_first(base['co2_fpb_ppm'])

# Basic sensor cleaning
base['flow_cfm'] = pd.to_numeric(base['flow_cfm'], errors='coerce')
base['co2_ppm_raw'] = pd.to_numeric(base['co2_ppm_raw'], errors='coerce')
base = base[(base['flow_cfm'].notna()) & (base['co2_ppm_raw'].notna())]
base = base[(base['co2_ppm_raw'] >= 250) & (base['co2_ppm_raw'] <= 5000)]
base = base[(base['flow_cfm'] >= 0)]

# 5) Assumptions: edit as needed
BMR_MJ_day = 7.73      # example adult from paper
M_met = 1.2            # sitting/office-like
RQ = 0.85
G_person = vco2_bmr_273k_101kpa(BMR_MJ_day, M_met, RQ=RQ)  # L/s-person

# Outdoor CO2 estimate from low percentile of observed indoor CO2
# (replace with measured outdoor CO2 if available)
C_out_ppm = float(base['co2_ppm_raw'].quantile(0.05))

# 6) Compute occupancy per measurement
CFM_TO_LPS = 0.47194745
base['Q_Lps'] = base['flow_cfm'] * CFM_TO_LPS
base['delta_co2_frac'] = (base['co2_ppm_raw'] - C_out_ppm) / 1_000_000.0
base['delta_co2_frac'] = base['delta_co2_frac'].clip(lower=0)
base['people_est'] = (base['Q_Lps'] * base['delta_co2_frac']) / G_person
base['people_est'] = base['people_est'].clip(lower=0)
base['people_est_rounded'] = base['people_est'].round().astype(int)

# Optional smoothing for operational interpretation
base['people_est_smooth_5min'] = (base.set_index('ts')['people_est']
                                 .rolling('5min', min_periods=1).median()
                                 .values)
base['people_est_smooth_5min_rounded'] = base['people_est_smooth_5min'].round().astype(int)

occupancy = base[['ts','flow_cfm','Q_Lps','co2_ppm_raw','people_est','people_est_rounded',
                  'people_est_smooth_5min','people_est_smooth_5min_rounded']].copy()
occupancy = occupancy.rename(columns={'co2_ppm_raw':'co2_ppm'})

print(f'Rows computed: {len(occupancy):,}')
print(f'G_person used: {G_person:.6f} L/s-person (BMR={BMR_MJ_day}, M={M_met}, RQ={RQ})')
print(f'Estimated C_out: {C_out_ppm:.1f} ppm')
print(occupancy.head(10).to_string(index=False))

Rows computed: 40,236
G_person used: 0.004486 L/s-person (BMR=7.73, M=1.2, RQ=0.85)
Estimated C_out: 404.0 ppm
                 ts    flow_cfm      Q_Lps  co2_ppm  people_est  people_est_rounded  people_est_smooth_5min  people_est_smooth_5min_rounded
2026-02-27 06:00:28 1099.784180 519.040339    443.0    4.512049                   5                4.512049                               5
2026-02-27 06:00:58 1087.170654 513.087418    443.0    4.460300                   4                4.486174                               4
2026-02-27 06:01:58 1092.324829 515.519918    442.0    4.366537                   4                4.460300                               4
2026-02-27 06:03:28 1116.523071 526.940216    444.0    4.698177                   5                4.486174                               4
2026-02-27 06:04:28 1109.863037 523.797030    444.0    4.670153                   5                4.512049                               5
2026-02-27 06:06:58 1102.431885 520.289917    443

In [11]:
# Save per-measurement occupancy estimates
out_csv = Path('/workspaces/CSC_4260_project/reports/room354_people_estimated_each_measurement.csv')
occupancy.to_csv(out_csv, index=False)
print(f'Saved: {out_csv}')

Saved: /workspaces/CSC_4260_project/reports/room354_people_estimated_each_measurement.csv


In [12]:
# Quick sanity summary
summary = occupancy['people_est_smooth_5min'].describe(percentiles=[0.25,0.5,0.75,0.9,0.95,0.99])
print(summary.to_string())

peak_row = occupancy.loc[occupancy['people_est_smooth_5min'].idxmax()]
print('\nPeak estimated occupancy (5-min median):')
print(peak_row[['ts','people_est_smooth_5min','flow_cfm','co2_ppm']].to_string())

count    40236.000000
mean         7.297345
std         11.147774
min          0.000000
25%          1.629299
50%          3.233483
75%          6.987704
90%         19.777144
95%         30.513324
99%         58.447021
max         97.876838

Peak estimated occupancy (5-min median):
ts                        2026-03-11 16:59:25
people_est_smooth_5min              97.876838
flow_cfm                           2230.13208
co2_ppm                                 815.0


## Source
- https://energy-models.com/internal-heat-gains-ihg
- https://www.asami.lt/media/heat-gains-calculations/

- https://pmc.ncbi.nlm.nih.gov/articles/PMC5666301/

- Moisture generation text provided by user (people, pets, plants; includes Eq. 1-3).
